#**TRABAJO DE VALIDACIÓN**
--------------------------------------------------------------------------------
* **Marcos Cortizo Albiñana**

* **Raquel Pérez Roa**

En esta fase se aplica el modelo final seleccionado, Random Forest, sobre un conjunto externo de evaluación que no incluye la variable objetivo. El dataset se preprocesa para reproducir la estructura utilizada durante el entrenamiento, corrigiendo valores categóricos redundantes, codificando las variables y asegurando que las columnas coincidan con las esperadas por el modelo.

Aunque el modelo fue entrenado con un conjunto en el que se eliminaron outliers, en el dataset de evaluación no se eliminan filas, ya que el enunciado exige obtener una predicción para todos los pacientes y mantener el orden original de ID_Paciente. Al no existir columna objetivo, tampoco se calculan métricas de rendimiento ni matriz de confusión.

In [ ]:
import pandas as pd
import joblib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_eval = pd.read_csv('/content/drive/MyDrive/UNIVERSIDAD/4 carrera/Proyecto Sistemas/muestra_evaluacion_grupo_6.csv')

##Preprocesado de datos

Normalización de etiquetas categóricas.

In [ ]:
data_eval['Exposicion_Biomasa'] = data_eval['Exposicion_Biomasa'].replace('ALTA', 'Alta')
data_eval['Exposicion_Biomasa'] = data_eval['Exposicion_Biomasa'].replace('baja ', 'Baja')
data_eval['Exposicion_Biomasa'] = data_eval['Exposicion_Biomasa'].replace('ninguna', 'Ninguna')
data_eval['Exposicion_Biomasa'] = data_eval['Exposicion_Biomasa'].replace('MODERADA', 'Media')

# Conversión de Exposicion_Biomasa a valores numéricos
data_eval['Exposicion_Biomasa'] = data_eval['Exposicion_Biomasa'].replace('Ninguna', 0)
data_eval['Exposicion_Biomasa'] = data_eval['Exposicion_Biomasa'].replace('Baja', 1)
data_eval['Exposicion_Biomasa'] = data_eval['Exposicion_Biomasa'].replace('Media', 2)
data_eval['Exposicion_Biomasa'] = data_eval['Exposicion_Biomasa'].replace('Alta', 3)

data_eval.head()

/tmp/ipykernel_5620/694977491.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_eval['Exposicion_Biomasa'] = data_eval['Exposicion_Biomasa'].replace('Alta', 3)


,FEV1_L,FVC_L,Ratio_FEV1_FVC,Flujo_Espiratorio_Max_L_min,Capacidad_Pulmonar_Total_L,Saturacion_O2_Reposo_porcentaje,Saturacion_O2_Esfuerzo_porcentaje,Nivel_Eosinofilos_cel_mcL,Distancia_Marcha_6min_m,Frecuencia_Respiratoria_rpm,Anios_Fumador,ID_Paciente,Prueba_Principal,Calibracion_Equipo,Exposicion_Biomasa
0,2.72,2.40,0.53,318.58,5.44,89.54,87.42,521.75,309.75,18.35,48,NEUMO-327780,Espirometría con Broncodilatador,OK,0
1,3.57,4.03,0.55,347.84,8.08,91.74,80.98,370.24,279.40,19.51,22,NEUMO-191227,Espirometría con Broncodilatador,OK,1
2,3.44,3.98,0.84,236.44,7.28,91.05,93.71,462.01,223.04,20.78,25,NEUMO-518786,Espirometría con Broncodilatador,OK,0
3,3.08,5.21,0.56,622.05,5.53,92.68,104.73,605.16,753.70,26.97,21,NEUMO-152346,Espirometría con Broncodilatador,OK,3
4,3.03,3.07,0.78,478.49,6.39,92.31,96.75,393.06,480.88,23.22,9,NEUMO-994326,Espirometría con Broncodilatador,OK,3


Guardado de columna de Identificación del paciente y eliminación de columnas constantes

In [ ]:
# Se guarda la columna ID_Paciente para conservar el orden original
ids_pacientes = data_eval['ID_Paciente'].copy()

# Eliminación de columnas constantes
data_eval = data_eval.loc[:, data_eval.nunique() > 1]

# Se elimina ID_Paciente para que no entre al modelo
X_eval = data_eval.drop(columns=['ID_Paciente'], errors='ignore')

X_eval.head()

,FEV1_L,FVC_L,Ratio_FEV1_FVC,Flujo_Espiratorio_Max_L_min,Capacidad_Pulmonar_Total_L,Saturacion_O2_Reposo_porcentaje,Saturacion_O2_Esfuerzo_porcentaje,Nivel_Eosinofilos_cel_mcL,Distancia_Marcha_6min_m,Frecuencia_Respiratoria_rpm,Anios_Fumador,Exposicion_Biomasa
0,2.72,2.40,0.53,318.58,5.44,89.54,87.42,521.75,309.75,18.35,48,0
1,3.57,4.03,0.55,347.84,8.08,91.74,80.98,370.24,279.40,19.51,22,1
2,3.44,3.98,0.84,236.44,7.28,91.05,93.71,462.01,223.04,20.78,25,0
3,3.08,5.21,0.56,622.05,5.53,92.68,104.73,605.16,753.70,26.97,21,3
4,3.03,3.07,0.78,478.49,6.39,92.31,96.75,393.06,480.88,23.22,9,3


###Importación el algoritmo

In [ ]:
modelo = joblib.load('/content/drive/MyDrive/UNIVERSIDAD/4 carrera/Proyecto Sistemas/random_forest_media_moda_sin_outliers.pkl')

In [ ]:
print("\nColumnas usadas para la predicción:")
print(X_eval.columns.tolist())

print("\nPrimeras filas del dataset que entra al modelo:")
display(X_eval.head())



Columnas usadas para la predicción:
['FEV1_L', 'FVC_L', 'Ratio_FEV1_FVC', 'Flujo_Espiratorio_Max_L_min', 'Capacidad_Pulmonar_Total_L', 'Saturacion_O2_Reposo_porcentaje', 'Saturacion_O2_Esfuerzo_porcentaje', 'Nivel_Eosinofilos_cel_mcL', 'Distancia_Marcha_6min_m', 'Frecuencia_Respiratoria_rpm', 'Anios_Fumador', 'Exposicion_Biomasa']

Primeras filas del dataset que entra al modelo:


,FEV1_L,FVC_L,Ratio_FEV1_FVC,Flujo_Espiratorio_Max_L_min,Capacidad_Pulmonar_Total_L,Saturacion_O2_Reposo_porcentaje,Saturacion_O2_Esfuerzo_porcentaje,Nivel_Eosinofilos_cel_mcL,Distancia_Marcha_6min_m,Frecuencia_Respiratoria_rpm,Anios_Fumador,Exposicion_Biomasa
0,2.72,2.40,0.53,318.58,5.44,89.54,87.42,521.75,309.75,18.35,48,0
1,3.57,4.03,0.55,347.84,8.08,91.74,80.98,370.24,279.40,19.51,22,1
2,3.44,3.98,0.84,236.44,7.28,91.05,93.71,462.01,223.04,20.78,25,0
3,3.08,5.21,0.56,622.05,5.53,92.68,104.73,605.16,753.70,26.97,21,3
4,3.03,3.07,0.78,478.49,6.39,92.31,96.75,393.06,480.88,23.22,9,3


In [ ]:
#se aplica el modelo
predicciones = modelo.predict(X_eval)

# Se asegura que las predicciones sean 0 o 1
predicciones = predicciones.astype(int)

print("Predicciones generadas:")
print(predicciones)


Predicciones generadas:
[1 1 0 1 1 0]


In [ ]:
resultado = pd.DataFrame({
    'ID_Paciente': ids_pacientes,
    'predicción': predicciones})

print("\nResultado final que se va a guardar:")
display(resultado)


Resultado final que se va a guardar:


,ID_Paciente,predicción
0,NEUMO-327780,1
1,NEUMO-191227,1
2,NEUMO-518786,0
3,NEUMO-152346,1
4,NEUMO-994326,1
5,NEUMO-937270,0


In [ ]:
resultado.to_csv('/content/drive/MyDrive/UNIVERSIDAD/4 carrera/Proyecto Sistemas/resultado_evaluacion_grupo_6.csv', index=False)

El modelo se ha aplicado correctamente al dataset de evaluación y ha generado una predicción para cada paciente, manteniendo el orden original de la columna ID_Paciente. El fichero final contiene únicamente las dos columnas solicitadas: ID_Paciente y predicción. Las predicciones se expresan en formato binario, donde 1 indica presencia de riesgo de exacerbación y 0 indica ausencia de riesgo. En la muestra obtenida, el modelo clasifica a la mayoría de los pacientes como casos con riesgo, aunque también identifica algunos pacientes sin riesgo.